In [1]:
from plotly.io import show
from sklearn.model_selection import train_test_split

from skfolio import Population, RiskMeasure
from skfolio.cluster import HierarchicalClustering, LinkageMethod
from skfolio.datasets import load_factors_dataset, load_sp500_dataset
from skfolio.distance import KendallDistance
from skfolio.optimization import EqualWeighted, HierarchicalRiskParity
from skfolio.preprocessing import prices_to_returns
from skfolio.prior import FactorModel

prices = load_sp500_dataset()
factor_prices = load_factors_dataset()

prices = prices["2014":]
factor_prices = factor_prices["2014":]

X, y = prices_to_returns(prices, factor_prices)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, shuffle=False)

In [2]:
model1 = HierarchicalRiskParity(
    risk_measure=RiskMeasure.CVAR, portfolio_params=dict(name="HRP-CVaR-Ward-Pearson")
)
model1.fit(X_train)
model1.weights_

array([0.05033705, 0.02773558, 0.05289115, 0.03632272, 0.059202  ,
       0.02483767, 0.03790179, 0.07464383, 0.03497807, 0.08622477,
       0.06308422, 0.04094166, 0.03144452, 0.08277551, 0.04421773,
       0.04807705, 0.02596219, 0.07596741, 0.0393462 , 0.06310889])

In [3]:
ptf1 = model1.predict(X_train)
ptf1.plot_contribution(measure=RiskMeasure.CVAR)

In [4]:
model1.hierarchical_clustering_estimator_.plot_dendrogram(heatmap=False)

In [5]:
fig = model1.hierarchical_clustering_estimator_.plot_dendrogram()
show(fig)

In [6]:
# To show this effect, let's create a second model with the single-linkage method:
model2 = HierarchicalRiskParity(
    risk_measure=RiskMeasure.CVAR,
    hierarchical_clustering_estimator=HierarchicalClustering(
        linkage_method=LinkageMethod.SINGLE,
    ),
    portfolio_params=dict(name="HRP-CVaR-Single-Pearson"),
)
model2.fit(X_train)

model2.hierarchical_clustering_estimator_.plot_dendrogram(heatmap=True)

In [7]:
model3 = HierarchicalRiskParity(
    risk_measure=RiskMeasure.CVAR,
    distance_estimator=KendallDistance(absolute=True),
    portfolio_params=dict(name="HRP-CVaR-Ward-Kendal"),
)
model3.fit(X_train)

model3.hierarchical_clustering_estimator_.plot_dendrogram(heatmap=True)

In [8]:
model4 = HierarchicalRiskParity(
    risk_measure=RiskMeasure.CVAR,
    prior_estimator=FactorModel(),
    portfolio_params=dict(name="HRP-CVaR-Factor-Model"),
)
model4.fit(X_train, y_train)

model4.hierarchical_clustering_estimator_.plot_dendrogram(heatmap=True)

In [9]:
bench = EqualWeighted()
bench.fit(X_train)
bench.weights_

array([0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05,
       0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05])

In [10]:
population_test = Population([])
for model in [model1, model2, model3, model4, bench]:
    population_test.append(model.predict(X_test))

population_test.plot_cumulative_returns()

In [11]:
population_test.plot_composition()

In [12]:
summary = population_test.summary()
summary.loc["Annualized Sharpe Ratio"]

HRP-CVaR-Ward-Pearson      0.86
HRP-CVaR-Single-Pearson    0.84
HRP-CVaR-Ward-Kendal       0.86
HRP-CVaR-Factor-Model      0.87
EqualWeighted              0.86
Name: Annualized Sharpe Ratio, dtype: object

In [13]:
summary 

,HRP-CVaR-Ward-Pearson,HRP-CVaR-Single-Pearson,HRP-CVaR-Ward-Kendal,HRP-CVaR-Factor-Model,EqualWeighted
Mean,0.081%,0.079%,0.079%,0.081%,0.084%
Annualized Mean,20.41%,19.81%,19.95%,20.29%,21.27%
Variance,0.022%,0.022%,0.022%,0.022%,0.024%
Annualized Variance,5.62%,5.51%,5.42%,5.47%,6.13%
Semi-Variance,0.011%,0.011%,0.011%,0.011%,0.012%
Annualized Semi-Variance,2.89%,2.82%,2.78%,2.78%,3.11%
Standard Deviation,1.49%,1.48%,1.47%,1.47%,1.56%
Annualized Standard Deviation,23.71%,23.48%,23.29%,23.38%,24.75%
Semi-Deviation,1.07%,1.06%,1.05%,1.05%,1.11%
Annualized Semi-Deviation,16.99%,16.78%,16.69%,16.66%,17.63%
